#Inference Optimization
In this notebook, I benchmark the basic transformers library forward pass for the TinyLlama-1.1B-Chat-v1.0 llm. I then iteratively optimize the inference via vllm and various techniques.

In [ ]:
%pip install -q transformers torch matplotlib
nvidia-smi

In [ ]:
import time
import statistics
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "GPU runtime not enabled"
print(torch.cuda.get_device_name(0))

# Load Model

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype, attn_implementation="sdpa").to("cuda")
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{model_name}: {n_params/1e6:.1f}M params, dtype = {dtype}")

# Timing Latency
Here, I establish functions to manually time the prefill latency (TTFT) and the decode latency (ITL samples, mean = TPOT)

In [ ]:
@torch.no_grad()
def single_trial(input_ids, max_new_tokens: int):
  """
  Runs a single forward pass and returns ttft (time to first token), itl (list of per step decode latencies),
  n_generated (number of tokens generated)

  """
  torch.cuda.synchronize()

  # prefill:
  t0 = time.perf_counter()
  outputs = model(input_ids=input_ids, use_cache = True)
  torch.cuda.synchronize()
  ttft = time.perf_counter() - t0

  past_key_values = outputs.past_key_values
  next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
  generated = [next_token.item()]

  # decode loop:
  itl = []
  for _ in range(max_new_tokens - 1):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    outputs = model(input_ids=next_token, past_key_values=past_key_values, use_cache = True)
    torch.cuda.synchronize()
    itl.append(time.perf_counter() - t0)

    past_key_values = outputs.past_key_values
    next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated.append(next_token.item())

    if next_token.item() == tokenizer.eos_token_id:
      break
  return ttft, itl, len(generated)


def pct(data, p):
  data = sorted(data)
  k = (len(data)-1) * (p/100)
  f, c = int(k), min(int(k)+1, len(data) - 1)
  return data[f] + (data[c] - data[f]) * (k-f)

# Single Profiling run
Warmup first since first few passes pay for cuDNN autotuning and kernel compilation (inflating numbers)


In [ ]:
def run_profile(prompt, max_new_tokens=128, n_trials=10, warmup=3):
  input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
  prompt_len = input_ids.shape[1]

  for _ in range(warmup):
    single_trial(input_ids, max_new_tokens=min(8, max_new_tokens))

  torch.cuda.reset_peak_memory_stats()

  all_ttft, all_itl, all_n = [], [], []
  for _ in range(n_trials):
    ttft, itl, n = single_trial(input_ids, max_new_tokens=max_new_tokens)
    all_ttft.append(ttft)
    all_itl.append(itl)
    all_n.append(n)

  peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

  return {
      "prompt_len": prompt_len,
      "ttft_samples": all_ttft,
      "itl_samples": all_itl,
      "n_generated_samples": all_n,
      "peak_mem_gb": peak_mem_gb,
  }


def report(results):
    ttft, itl = results["ttft_samples"], results["itl_samples"]
    prompt_len = results["prompt_len"]
    ttft_mean, tpot_mean = statistics.mean(ttft), statistics.mean(itl)

    print(f"Prompt length:     {prompt_len} tokens")
    print(f"Trials:            {len(ttft)}  |  ITL samples pooled: {len(itl)}")
    print(f"Peak GPU memory:   {results['peak_mem_gb']:.2f} GB\n")

    print("---- TTFT (prefill) ----")
    print(f"  mean: {ttft_mean*1000:.2f} ms   p50: {pct(ttft,50)*1000:.2f} ms   "
          f"p90: {pct(ttft,90)*1000:.2f} ms   p99: {pct(ttft,99)*1000:.2f} ms")
    print(f"  prefill throughput: {prompt_len/ttft_mean:.1f} tokens/sec\n")

    print("---- TPOT / ITL (decode) ----")
    print(f"  mean (TPOT): {tpot_mean*1000:.2f} ms/token   p50: {pct(itl,50)*1000:.2f} ms   "
          f"p90: {pct(itl,90)*1000:.2f} ms   p99: {pct(itl,99)*1000:.2f} ms")
    print(f"  decode throughput: {1/tpot_mean:.1f} tokens/sec\n")

    avg_n = statistics.mean(results["n_generated_samples"])
    e2e = ttft_mean + tpot_mean * (avg_n - 1)
    print(f"---- Composite ----")
    print(f"  avg output tokens: {avg_n:.1f}")
    print(f"  reconstructed E2E latency: {e2e*1000:.2f} ms")

In [ ]:
results = run_profile(
    prompt = "Explain speculative decoding in simple terms",
    max_new_tokens = 64,
    n_trials = 10,
    warmup = 3,
)
report(results)

# ITL Distribution
Plotting the distribution of time for each output token (TPOT = mean(ITL))

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist([x * 1000 for x in results["itl_samples"]], bins=40)
plt.axvline(statistics.mean(results["itl_samples"]) * 1000, color="red", linestyle="--", label="mean (TPOT)")
plt.xlabel("Inter-token latency (ms)")
plt.ylabel("count")
plt.title("Decode step latency distribution")
plt.legend()
plt.show()

# TTFT vs Prompt Length
Prefill is compute bound and attention is O(n^2) computational time complexity in terms of sequence length so TTFT (prefill time) should grow linearly/super linearly with prompt length. We validate below

In [ ]:
prompt_lengths_to_test = [32,128,512,1024]
base_text = "The quick brown fox jumps over the lazy dog. " * 200

ttft_vs_len = []
for target_len in prompt_lengths_to_test:
  ids = tokenizer(base_text, return_tensors="pt").input_ids[:, :target_len].to("cuda")
  for _ in range(2):
    single_trial(ids, max_new_tokens=4)
  trial_ttfts = []
  for _ in range(8):
    ttft, _, _ = single_trial(ids, max_new_tokens=4)
    trial_ttfts.append(ttft)
  mean_ttft = statistics.mean(trial_ttfts)
  ttft_vs_len.append(mean_ttft)
  print(f"prompt_len={target_len:>5}  TTFT mean={mean_ttft*1000:.2f} ms  "
          f"throughput={target_len/mean_ttft:.1f} tok/s")

plt.figure(figsize=(7, 4))
plt.plot(prompt_lengths_to_test, [t * 1000 for t in ttft_vs_len], marker="o")
plt.xlabel("Prompt length (tokens)")
plt.ylabel("TTFT (ms)")
plt.title("TTFT vs. prompt length")
plt.grid(alpha=0.3)
plt.show()

# TPOT vs. Context Length
Decode is memory/bandwidth bound since each steps reads the growing KV cache plus model weights. Therefore, TPOT should slowly increase as context grows unlike TTFT's steeper prefill scaling

In [ ]:
context_lengths_to_test = [32, 256, 1024, 2048]
tpot_vs_ctx = []

for ctx_len in context_lengths_to_test:
    ids = tokenizer(base_text, return_tensors="pt").input_ids[:, :ctx_len].to("cuda")
    for _ in range(2):
        single_trial(ids, max_new_tokens=8)
    _, itl, _ = single_trial(ids, max_new_tokens=40)
    mean_tpot = statistics.mean(itl)
    tpot_vs_ctx.append(mean_tpot)
    print(f"context_len={ctx_len:>5}  TPOT mean={mean_tpot*1000:.3f} ms/token  "
          f"throughput={1/mean_tpot:.1f} tok/s")

plt.figure(figsize=(7, 4))
plt.plot(context_lengths_to_test, [t * 1000 for t in tpot_vs_ctx], marker="o", color="darkorange")
plt.xlabel("Context length at start of decode (tokens)")
plt.ylabel("TPOT (ms/token)")
plt.title("TPOT vs. context length")
plt.grid(alpha=0.3)
plt.show()
